In [1]:
#Importing Libraries-

import sqlite3
import pandas as pd
import os

## Step 1-Connecting Database

In [2]:

def create_database(db_name: str = 'superstore.db') -> sqlite3.Connection:
    """Create SQLite database connection with foreign key support."""
    conn = sqlite3.connect(db_name)
    conn.execute("PRAGMA foreign_keys = ON")
    return conn

conn = create_database()
cursor = conn.cursor()
print("Database connected: superstore.db")

Database connected: superstore.db


## Step 2 - Load CSVs into superstore_raw

In [3]:
def load_csvs(conn: sqlite3.Connection) -> None:
    """
    Load customer, order, product CSVs.
    Merge into one superstore_raw table.
    """
    base_path = os.path.dirname(os.path.abspath('/Users/jain/Celebal_Internship_Assignment/Assignment3/Data/*.csv'))

    customers_df = pd.read_csv(os.path.join(base_path, 'customer.csv'))
    orders_df    = pd.read_csv(os.path.join(base_path, 'order.csv'))
    products_df  = pd.read_csv(os.path.join(base_path, 'product.csv'))

    # Merge all into one table
    orders_customers = orders_df.merge(customers_df, on='customer_id', how='left')
    superstore_raw   = orders_customers.merge(products_df, how='cross')

    # Load into SQLite table
    superstore_raw.to_sql('superstore_raw', conn, if_exists='replace', index=False)


load_csvs(conn)
print("Superstore_raw Created")

Superstore_raw Created


## Step 3 - Create Structured Tables

In [4]:
def create_tables(cursor: sqlite3.Cursor) -> None:
    """Create customers, products, orders with constraints."""

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS customers (
            customer_id  INTEGER      PRIMARY KEY,
            first_name   VARCHAR(50)  NOT NULL,
            last_name    VARCHAR(50)  NOT NULL,
            email        VARCHAR(100) UNIQUE NOT NULL,
            city         VARCHAR(50)  NOT NULL,
            state        VARCHAR(50)  NOT NULL,
            join_date    DATE         NOT NULL,
            is_premium   BOOLEAN      DEFAULT FALSE
        )
    """)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS products (
            product_id   INTEGER       PRIMARY KEY,
            product_name VARCHAR(100)  NOT NULL,
            category     VARCHAR(50)   NOT NULL,
            brand        VARCHAR(50)   NOT NULL,
            unit_price   DECIMAL(10,2) NOT NULL CHECK (unit_price > 0),
            stock_qty    INTEGER       NOT NULL DEFAULT 0 CHECK (stock_qty >= 0)
        )
    """)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS orders (
            order_id     INTEGER       PRIMARY KEY,
            customer_id  INTEGER       NOT NULL,
            order_date   DATE          NOT NULL,
            status       VARCHAR(20)   NOT NULL DEFAULT 'Pending'
                         CHECK (status IN ('Pending','Shipped','Delivered','Cancelled')),
            total_amount DECIMAL(12,2) NOT NULL CHECK (total_amount >= 0),
            FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
        )
    """)

    conn.commit()
    print("Tables created: customers, products, orders")

create_tables(cursor)

Tables created: customers, products, orders


## Step 4 - Create Indexes

In [5]:
def create_indexes(cursor: sqlite3.Cursor) -> None:
    """Create indexes on high-filter columns for query performance."""
    indexes = [
        "CREATE INDEX IF NOT EXISTS idx_customers_city     ON customers(city)",
        "CREATE INDEX IF NOT EXISTS idx_customers_state    ON customers(state)",
        "CREATE INDEX IF NOT EXISTS idx_products_category  ON products(category)",
        "CREATE INDEX IF NOT EXISTS idx_orders_customer    ON orders(customer_id)",
        "CREATE INDEX IF NOT EXISTS idx_orders_date        ON orders(order_date)",
    ]
    for idx in indexes:
        cursor.execute(idx)
    conn.commit()
    print(f"{len(indexes)} indexes created")

create_indexes(cursor)

5 indexes created


## Step 5 - Data Insertion


In [6]:
def insert_data(cursor: sqlite3.Cursor) -> None:

    # Insert customers
    cursor.execute("""
        INSERT OR IGNORE INTO customers
            (customer_id, first_name, last_name, email, city, state, join_date, is_premium)
        SELECT DISTINCT
            customer_id, first_name, last_name, email, city, state, join_date, is_premium
        FROM superstore_raw
    """)

    # Insert products
    cursor.execute("""
        INSERT OR IGNORE INTO products
            (product_id, product_name, category, brand, unit_price, stock_qty)
        SELECT DISTINCT
            product_id, product_name, category, brand, unit_price, stock_qty
        FROM superstore_raw
    """)

    # Insert orders (after customers due to Foreign Key constraint)
    cursor.execute("""
        INSERT OR IGNORE INTO orders
            (order_id, customer_id, order_date, status, total_amount)
        SELECT DISTINCT
            order_id, customer_id, order_date, status, total_amount
        FROM superstore_raw
    """)

    conn.commit()
    print("Data inserted into all tables")

insert_data(cursor)

Data inserted into all tables


## Step 6 - Verify Data

In [7]:
# Preview all 3 tables
display(pd.read_sql("SELECT * FROM customers", conn))

display(pd.read_sql("SELECT * FROM products", conn))

display(pd.read_sql("SELECT * FROM orders", conn))

,customer_id,first_name,last_name,email,city,state,join_date,is_premium
0,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,1
1,102,Priya,Patel,priya.p@email.com,Ahmedabad,Gujarat,2024-02-20,0
2,103,Rohan,Gupta,rohan.g@email.com,Delhi,Delhi,2024-03-10,1
3,104,Sneha,Reddy,sneha.r@email.com,Hyderabad,Telangana,2024-04-05,0
4,105,Vikram,Singh,vikram.s@email.com,Jaipur,Rajasthan,2024-05-12,1
5,106,Ananya,Iyer,ananya.i@email.com,Chennai,Tamil Nadu,2024-06-18,0
6,107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,1
7,108,Divya,Nair,divya.n@email.com,Kochi,Kerala,2024-08-30,0


,product_id,product_name,category,brand,unit_price,stock_qty
0,201,Wireless Earbuds,Electronics,BoAt,1499,250
1,202,Cotton T-Shirt,Clothing,Levi's,799,500
2,203,Smart Watch,Electronics,Noise,2999,150
3,204,Running Shoes,Clothing,Nike,4599,120
4,205,Bluetooth Speaker,Electronics,JBL,3499,200
5,206,Bedsheet Set,Home,Spaces,1299,300
6,207,Laptop Stand,Electronics,AmazonBasics,899,180


,order_id,customer_id,order_date,status,total_amount
0,1001,101,2024-08-01,Delivered,4498
1,1002,102,2024-08-03,Delivered,799
2,1003,103,2024-08-05,Shipped,7498
3,1004,101,2024-08-10,Delivered,3499
4,1005,104,2024-08-12,Cancelled,2999
5,1006,105,2024-08-15,Delivered,5898
6,1007,106,2024-08-18,Pending,1299
7,1008,103,2024-08-20,Delivered,899
8,1009,107,2024-08-25,Shipped,6098
9,1010,108,2024-08-28,Delivered,1598


## Step 7 - SQL Analysis Queries

__Find all orders where sales are greater than the average sales.__

In [8]:
pd.read_sql("""SELECT * FROM orders WHERE total_amount > (SELECT AVG(total_amount) FROM orders)
            """,conn)

,order_id,customer_id,order_date,status,total_amount
0,1001,101,2024-08-01,Delivered,4498
1,1003,103,2024-08-05,Shipped,7498
2,1006,105,2024-08-15,Delivered,5898
3,1009,107,2024-08-25,Shipped,6098


__Find the highest sales order for each customer.__

In [9]:
pd.read_sql("""SELECT*
FROM superstore_raw s1
WHERE total_amount = (
    SELECT MAX(total_amount)
    FROM superstore_raw s2
    WHERE s1.customer_id = s2.customer_id
)
""",conn);

__Calculate total sales for each customer.__

In [10]:
pd.read_sql(""" WITH customer_sales AS (
    SELECT customer_id, SUM(total_amount) AS total_sales
    FROM superstore_raw GROUP BY customer_id
)
SELECT c.first_name, c.last_name AS customer_name,
       cs.total_sales
FROM customer_sales cs
JOIN customers c ON cs.customer_id = c.customer_id
ORDER BY cs.total_sales DESC
""",conn)

,first_name,customer_name,total_sales
0,Rohan,Gupta,58779.0
1,Aarav,Sharma,55979.0
2,Karan,Mehta,42686.0
3,Vikram,Singh,41286.0
4,Sneha,Reddy,20993.0
5,Divya,Nair,11186.0
6,Ananya,Iyer,9093.0
7,Priya,Patel,5593.0


__Find customers whose total sales are above average.__

In [11]:
pd.read_sql("""WITH customer_sales AS (
    SELECT
        customer_id,
        SUM(total_amount) AS total_sales
    FROM superstore_raw
    GROUP BY customer_id
)
SELECT *
FROM customer_sales
WHERE total_sales > (
    SELECT AVG(total_sales)
    FROM customer_sales
);
""",conn)

,customer_id,total_sales
0,101,55979.0
1,103,58779.0
2,105,41286.0
3,107,42686.0


__Rank all customers based on total sales.__

In [12]:
pd.read_sql("""
            SELECT customer_id, SUM(total_amount) AS total_sales,
               DENSE_RANK() OVER (ORDER BY SUM(total_amount) DESC) AS rank
                FROM superstore_raw 
                GROUP BY customer_id 
            """,conn)

,customer_id,total_sales,rank
0,103,58779.0,1
1,101,55979.0,2
2,107,42686.0,3
3,105,41286.0,4
4,104,20993.0,5
5,108,11186.0,6
6,106,9093.0,7
7,102,5593.0,8


__Assign row numbers to each order within a customer.__

In [13]:
pd.read_sql("""SELECT 
    customer_id,
    order_id,
    ROW_NUMBER() OVER (
        PARTITION BY customer_id
        ORDER BY order_id
    ) AS order_row_num
FROM superstore_raw;
""",conn)

,customer_id,order_id,order_row_num
0,101,1001,1
1,101,1001,2
2,101,1001,3
3,101,1001,4
4,101,1001,5
...,...,...,...
65,108,1010,3
66,108,1010,4
67,108,1010,5
68,108,1010,6


__Display top 3 customers based on total sales.__

In [23]:
pd.read_sql("""WITH ranked AS (
        SELECT first_name,last_name AS customer_name,
               SUM(total_amount) AS total_sales,
               DENSE_RANK() OVER (ORDER BY SUM(total_amount) DESC) AS customer_rank
        FROM superstore_raw
        GROUP BY customer_id
    )
    SELECT * FROM ranked WHERE customer_rank <= 3
                """,conn)
                

,first_name,customer_name,total_sales,customer_rank
0,Rohan,Gupta,58779.0,1
1,Aarav,Sharma,55979.0,2
2,Karan,Mehta,42686.0,3


__ Combined Query   
Write one query that shows:     
Customer Name  
Total Sales  
Rank  __

In [15]:
pd.read_sql("""
    WITH customer_sales AS (
        SELECT customer_id, SUM(total_amount) AS total_sales
        FROM superstore_raw
        GROUP BY customer_id
    )
    SELECT c.first_name,c.last_name AS customer_name,
           cs.total_sales,
           DENSE_RANK() OVER (ORDER BY cs.total_sales DESC) AS sales_rank
    FROM customer_sales cs
    JOIN customers c ON cs.customer_id = c.customer_id
    ORDER BY sales_rank
""", conn)

,first_name,customer_name,total_sales,sales_rank
0,Rohan,Gupta,58779.0,1
1,Aarav,Sharma,55979.0,2
2,Karan,Mehta,42686.0,3
3,Vikram,Singh,41286.0,4
4,Sneha,Reddy,20993.0,5
5,Divya,Nair,11186.0,6
6,Ananya,Iyer,9093.0,7
7,Priya,Patel,5593.0,8


__Who are the top 5 customers?__

In [16]:
pd.read_sql("""SELECT DISTINCT first_name,last_name FROM superstore_raw 
                ORDER BY total_amount DESC
                LIMIT 5;
                """,conn)

,first_name,last_name
0,Rohan,Gupta
1,Karan,Mehta
2,Vikram,Singh
3,Aarav,Sharma
4,Sneha,Reddy


__Who are the bottom 5 customers?__

In [17]:
pd.read_sql("""SELECT DISTINCT first_name,last_name FROM superstore_raw
                ORDER BY
                total_amount
                LIMIT 5;
                """,conn)

,first_name,last_name
0,Priya,Patel
1,Ananya,Iyer
2,Divya,Nair
3,Sneha,Reddy
4,Aarav,Sharma


__Which customers made only one order?__ 

In [18]:
 pd.read_sql(""" SELECT c.first_name || ' ' || c.last_name AS customer_name,
           COUNT(o.order_id)   AS order_count,
           SUM(o.total_amount) AS total_spent
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY o.customer_id
    HAVING COUNT(o.order_id) = 1
    ORDER BY total_spent DESC
                 """,conn)

,customer_name,order_count,total_spent
0,Karan Mehta,1,6098
1,Vikram Singh,1,5898
2,Sneha Reddy,1,2999
3,Divya Nair,1,1598
4,Ananya Iyer,1,1299
5,Priya Patel,1,799


__Which customers have above-average sales?__

In [19]:
pd.read_sql("""SELECT DISTINCT first_name,last_name FROM superstore_raw 
                Where total_amount>(Select AVG(total_amount) from superstore_raw)
                """,conn)

,first_name,last_name
0,Aarav,Sharma
1,Rohan,Gupta
2,Vikram,Singh
3,Karan,Mehta


__What is the highest order value per customer?__

In [20]:
pd.read_sql("""SELECT first_name,last_name, MAX(total_amount) AS highest_order
FROM superstore_raw GROUP BY customer_id ORDER BY highest_order DESC
                """,conn)

,first_name,last_name,highest_order
0,Rohan,Gupta,7498.0
1,Karan,Mehta,6098.0
2,Vikram,Singh,5898.0
3,Aarav,Sharma,4498.0
4,Sneha,Reddy,2999.0
5,Divya,Nair,1598.0
6,Ananya,Iyer,1299.0
7,Priya,Patel,799.0


## Summary

| Step | What was done |
|------|---------------|
| Database | Single `superstore.db` — clean, no conflicts |
| Staging | CSVs merged into `superstore_raw` |
| Schema | 3 normalized tables with Primary Key, Foreign Key, CHECK, UNIQUE constraints |
| Indexes | 5 indexes on high-filter columns |
| Insert | `SELECT DISTINCT` + `INSERT OR IGNORE` — duplicate safe |
| Queries | 6 queries: subquery, JOIN, GROUP BY, CASE, window function |


In [ ]:
conn.close()
print("Connection closed")